Load and Prepare the Dataset

In [1]:
import pandas as pd

# Load the CSV
df = pd.read_csv(r"C:\Users\sun99\Desktop\Intaj\Model\Translate\arb_fr translate\Darija Latin+Arabic detection fine_tune\darija_dataset.csv")

# Keep only the needed columns
df = df[["text", "lang"]]

# Map languages to integers
label_mapping = {
    "darija_latin": 0,
    "darija_arabic": 1,
    "not_darija": 2
}
df["label"] = df["lang"].map(label_mapping)

# Verify no missing labels after mapping
if df["label"].isnull().any():
    print(" Warning: Some rows have an unexpected 'lang' value!")
    print(df[df["label"].isnull()]["lang"].value_counts())

# Check class distribution
print(df["label"].value_counts())
print(df.head())


label
2    1984
0     978
1     707
Name: count, dtype: int64
                      text          lang  label
0              labas 3lik?  darija_latin      0
1    wach rak mlih wla la?  darija_latin      0
2      rani mcharja bzaaaf  darija_latin      0
3  ghedwa ndirou barbekiou  darija_latin      0
4      3lach matji ma3ana?  darija_latin      0


Remove Duplicates and NaNs

In [2]:
# Drop empty rows
df.dropna(subset=["text", "lang"], inplace=True)

# Drop duplicate texts
df.drop_duplicates(subset=["text"], inplace=True)

# Reset index
df.reset_index(drop=True, inplace=True)


Save Cleaned Version

In [3]:
df.to_csv("cleaned_darija_dataset.csv", index=False,encoding="utf-8-sig")
print(df.head())

                      text          lang  label
0              labas 3lik?  darija_latin      0
1    wach rak mlih wla la?  darija_latin      0
2      rani mcharja bzaaaf  darija_latin      0
3  ghedwa ndirou barbekiou  darija_latin      0
4      3lach matji ma3ana?  darija_latin      0


 Base Model & Tokenizer

In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "xlm-roberta-base"

# Load tokenizer and model for 3-class classification
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)


c:\Users\sun99\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Tokenize & Prepare Dataset for Training

In [5]:
from datasets import Dataset
from transformers import AutoTokenizer

# Load cleaned CSV
df = pd.read_csv("cleaned_darija_dataset.csv")

# Convert to HuggingFace Dataset with only needed columns
dataset = Dataset.from_pandas(df[["text", "label"]])

# Tokenize function for batched tokenization
def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

# Apply tokenization
tokenized_dataset = dataset.map(tokenize, batched=True)

# Split into train and test sets (80/20)
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.2)

# Set format for PyTorch tensors including labels
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# Quick check on splits and keys
print(tokenized_dataset)


Map: 100%|██████████| 1968/1968 [00:00<00:00, 5880.47 examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 1574
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 394
    })
})


Set Up the Trainer and Fine-Tune

In [6]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

# Define compute_metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')  # use 'macro' for multiclass
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

# Training arguments
training_args = TrainingArguments(
    output_dir="./darija_detector_model",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)


trainer.train()


c:\Users\sun99\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
  0%|          | 0/297 [00:00<?, ?it/s]c:\Users\sun99\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
  3%|▎         | 10/297 [02:36<1:37:12, 20.32s/it]

{'loss': 1.0426, 'grad_norm': 4.850101947784424, 'learning_rate': 1.932659932659933e-05, 'epoch': 0.1}


  7%|▋         | 20/297 [07:59<5:36:22, 72.86s/it]

{'loss': 0.9388, 'grad_norm': 8.940988540649414, 'learning_rate': 1.8653198653198653e-05, 'epoch': 0.2}


 10%|█         | 30/297 [09:39<44:06,  9.91s/it]  

...

100%|██████████| 297/297 [51:57<00:00,  7.60s/it]c:\Users\sun99\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
                                                 
100%|██████████| 297/297 [52:54<00:00,  7.60s/it]

{'eval_loss': 0.01585184782743454, 'eval_accuracy': 0.9974619289340102, 'eval_precision': 0.99822695035461, 'eval_recall': 0.99625468164794, 'eval_f1': 0.9972278719397364, 'eval_runtime': 46.0961, 'eval_samples_per_second': 8.547, 'eval_steps_per_second': 0.542, 'epoch': 3.0}


100%|██████████| 297/297 [53:03<00:00, 10.72s/it]

{'train_runtime': 3183.0667, 'train_samples_per_second': 1.483, 'train_steps_per_second': 0.093, 'train_loss': 0.13135278384869148, 'epoch': 3.0}


TrainOutput(global_step=297, training_loss=0.13135278384869148, metrics={'train_runtime': 3183.0667, 'train_samples_per_second': 1.483, 'train_steps_per_second': 0.093, 'total_flos': 310605389627904.0, 'train_loss': 0.13135278384869148, 'epoch': 3.0})

Save the Model

In [12]:
model.save_pretrained("./darija_detector_model")
tokenizer.save_pretrained("./darija_detector_model")


('./darija_detector_model\\tokenizer_config.json',
 './darija_detector_model\\special_tokens_map.json',
 './darija_detector_model\\tokenizer.json')

Testing

In [13]:
from transformers import pipeline

# Load the trained model and tokenizer
model_path = "./darija_detector_model"  # or wherever you saved it
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)

# Try on new examples
texts = [
    "كي راك خويا؟",  # darija
    "How are you doing today?",  # not darija
    "أنا جيت البارح مع الليل",  # darija
    "Je suis très fatigué aujourd'hui.",  # not darija
    "machi kamel mli7a.",
]

for text in texts:
    pred = classifier(text)
    print(f" Text: {text}\n Prediction: {pred}\n")


 Text: كي راك خويا؟
 Prediction: [{'label': 'LABEL_1', 'score': 0.9896079897880554}]

 Text: How are you doing today?
 Prediction: [{'label': 'LABEL_2', 'score': 0.9967026114463806}]

 Text: أنا جيت البارح مع الليل
 Prediction: [{'label': 'LABEL_1', 'score': 0.9937113523483276}]

 Text: Je suis très fatigué aujourd'hui.
 Prediction: [{'label': 'LABEL_2', 'score': 0.9981468915939331}]

 Text: machi kamel mli7a.
 Prediction: [{'label': 'LABEL_0', 'score': 0.9977092742919922}]

